# Episode 15 — AXI DMA: moving data in bulk

MMIO moves one 32-bit word per Python call. To give a hardware block a million samples
you need **DMA** (direct memory access): the AXI DMA block reads a buffer from DDR memory and
streams it into the PL, or takes a stream from the PL and writes it to memory, while the
processor does something else.

This overlay connects the DMA to itself through a FIFO, so whatever we send comes back:

```
 DDR buffer --> DMA (MM2S) --stream--> FIFO --stream--> DMA (S2MM) --> DDR buffer
```

**Two things that are new:**
* The DMA needs **physically contiguous** memory. `pynq.allocate()` gives you a NumPy
  array in such memory; ordinary NumPy arrays will not work.
* The DMA reaches DDR through the processor's **HP0** port, a fast path to memory that
  doesn't go through the CPU.

Build: `vivado -mode batch -source dma_overlay.tcl -tclargs xc7z020clg400-1 work overlays/z20`

In [ ]:
from pynq import Overlay, allocate
from pynq.overlays.base import board_variant
import numpy as np
import time

chip = {"EDGE-Z10": "z10", "EDGE-Z20": "z20"}[board_variant()]
ol = Overlay("overlays/{}/dma_overlay.bit".format(chip))
dma = ol.axi_dma
print(type(dma).__name__, "at 0x{:08X}".format(ol.ip_dict["axi_dma"]["phys_addr"]))

## 1. Send an array and get it back

Start the receive side first, so it is ready when the data comes out of the FIFO.

In [ ]:
n = 1000
send = allocate(shape=(n,), dtype=np.uint32)
recv = allocate(shape=(n,), dtype=np.uint32)
send[:] = np.arange(n, dtype=np.uint32) * 3

print("send buffer at physical address 0x{:08X}".format(send.physical_address))

dma.recvchannel.transfer(recv)
dma.sendchannel.transfer(send)
dma.sendchannel.wait()
dma.recvchannel.wait()

print("first values back:", recv[:8])
print("all equal:", np.array_equal(send, recv))

## 2. How fast is it?

The stream is 32 bits wide at 100 MHz, so the limit is 400 MB/s. Small transfers are slower
because each one costs a fixed set-up time in Python.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

sizes = [2 ** k for k in range(8, 21, 2)]           # 256 words ... 1 M words (4 MB)
speeds = []
for words in sizes:
    a = allocate(shape=(words,), dtype=np.uint32)
    b = allocate(shape=(words,), dtype=np.uint32)
    a[:] = np.random.randint(0, 2 ** 32, size=words, dtype=np.uint64).astype(np.uint32)
    t0 = time.perf_counter()
    dma.recvchannel.transfer(b)
    dma.sendchannel.transfer(a)
    dma.sendchannel.wait()
    dma.recvchannel.wait()
    seconds = time.perf_counter() - t0
    assert np.array_equal(a, b)
    speeds.append(words * 4 / seconds / 1e6)
    a.freebuffer()
    b.freebuffer()

plt.semilogx([s * 4 for s in sizes], speeds, "o-")
plt.xlabel("transfer size (bytes)")
plt.ylabel("MB/s")
plt.grid(True)
plt.show()
for s, v in zip(sizes, speeds):
    print("{:>10,d} bytes  {:6.1f} MB/s".format(s * 4, v))

## 3. Compare with MMIO

For comparison, this is how fast Python moves 32-bit words one at a time with MMIO.
It reads the DMA's status register, which is harmless, 20 000 times.

In [ ]:
from pynq import MMIO

regs = MMIO(ol.ip_dict["axi_dma"]["phys_addr"], 0x100)
words = 20000
t0 = time.perf_counter()
for i in range(words):
    regs.read(0x04)              # the DMA's status register
seconds = time.perf_counter() - t0
print("MMIO: {:.2f} MB/s ({:.1f} µs per word)".format(words * 4 / seconds / 1e6, seconds / words * 1e6))

## 4. Free the buffers

Contiguous memory is limited; give it back when you are done.

In [ ]:
send.freebuffer()
recv.freebuffer()

## Try it yourself

Send a `float32` array of 10 000 random numbers through the loopback and check that the
values come back unchanged. The DMA only moves bytes; it does not care about the type.

In [ ]:
# your code here